# Notebook 22 — Castalia Bench Capstone

[![Open in Colab](https://colab.research.google.com/assets/colab-badge.svg)](https://colab.research.google.com/github/cyruslayo/castalia/blob/main/evals/22_castalia_bench_capstone.ipynb)

This capstone unifies the full evals course around **Castalia Bench**: a from-scratch benchmark harness for comparing three kinds of local LLM systems on the same workload:

- a prompt-only baseline
- a tiny retrieval-augmented pipeline
- a simple tool-using agent workflow

The goal is not to admire one lucky answer. The goal is to define a compact benchmark, run the systems side by side, score them transparently, inspect their failures, and finish with a reusable local report.

## 🎯 Learning Objectives

By the end of this notebook, you will:

- Understand **What you will build**
- Understand **How the earlier notebooks feed this capstone**
- Understand **Step 1 — Helpers and artifact directory**
- Understand **Step 2 — Build a compact local corpus**
- Understand **Step 3 — Define the benchmark tasks**


## What you will build

- a compact benchmark dataset with multiple task types
- a local document corpus and FAISS-backed retrieval index
- three systems under test built with plain Python helpers
- quality, groundedness, latency, cost-proxy, and success metrics
- comparison tables, failure buckets, and saved benchmark artifacts
- a final benchmark report written by the local open-source model

## How the earlier notebooks feed this capstone

This notebook is meant to feel like the natural end of the course:

- **01–04** gave us benchmark design, dataset construction, metrics, and failure buckets
- **05–08** taught deterministic and rubric-style prompt evaluation
- **09–12** added retrieval metrics, groundedness, citation checks, and RAG ablations
- **13–17** added tool-use success, trajectory inspection, cost/latency analysis, and robustness thinking
- **18–21** added regression discipline, safety framing, human review patterns, and experiment reporting

Now we put those ideas together in one small but complete benchmark harness.

In [ ]:
# --- Google Colab Setup ---
!pip install -q "transformers>=4.51.0" accelerate bitsandbytes torch sentence-transformers faiss-cpu numpy

import csv
import json
import math
import random
import statistics
import time
from pathlib import Path
from typing import Any, Dict, List

import faiss
import numpy as np
import torch
from google.colab import drive
from sentence_transformers import SentenceTransformer
from transformers import AutoModelForCausalLM, AutoTokenizer, BitsAndBytesConfig

drive.mount('/content/drive')
CACHE_DIR = "/content/drive/MyDrive/models"
MODEL_NAME = "Qwen/Qwen3-8B"
EMBEDDING_MODEL_NAME = "BAAI/bge-base-en-v1.5"

quantization_config = BitsAndBytesConfig(
    load_in_4bit=True,
    bnb_4bit_compute_dtype=torch.bfloat16,
    bnb_4bit_quant_type="nf4",
)

tokenizer = AutoTokenizer.from_pretrained(MODEL_NAME, cache_dir=CACHE_DIR)
if tokenizer.pad_token_id is None:
    tokenizer.pad_token_id = tokenizer.eos_token_id

model = AutoModelForCausalLM.from_pretrained(
    MODEL_NAME,
    device_map="auto",
    quantization_config=quantization_config,
    cache_dir=CACHE_DIR,
    torch_dtype="auto",
)

embed_model = SentenceTransformer(EMBEDDING_MODEL_NAME, cache_folder=CACHE_DIR)

def generate(messages, max_new_tokens=512, temperature=0.7, do_sample=True, top_k=20):
    if isinstance(messages, str):
        messages = [{"role": "user", "content": messages}]

    prompt = tokenizer.apply_chat_template(
        messages,
        tokenize=False,
        add_generation_prompt=True,
        enable_thinking=False,
    )
    inputs = tokenizer([prompt], return_tensors="pt").to(model.device)
    output_ids = model.generate(
        **inputs,
        max_new_tokens=max_new_tokens,
        temperature=temperature if do_sample else None,
        do_sample=do_sample,
        top_k=top_k,
        pad_token_id=tokenizer.pad_token_id,
    )
    generated_ids = output_ids[0][inputs.input_ids.shape[1]:]
    return tokenizer.decode(generated_ids, skip_special_tokens=True).strip()

def embed_texts(texts, batch_size=32):
    if isinstance(texts, str):
        texts = [texts]
    return embed_model.encode(
        list(texts),
        batch_size=batch_size,
        normalize_embeddings=True,
        convert_to_numpy=True,
    )

def build_faiss_index(texts):
    texts = list(texts)
    embeddings = embed_texts(texts).astype(np.float32)
    index = faiss.IndexFlatIP(embeddings.shape[1])
    index.add(embeddings)
    return {
        "texts": texts,
        "embeddings": embeddings,
        "index": index,
    }

def search_index(query, store, k=5):
    k = min(k, len(store["texts"]))
    if k <= 0:
        return []

    query_vector = embed_texts([query]).astype(np.float32)
    scores, indices = store["index"].search(query_vector, k)
    results = []
    for score, idx in zip(scores[0], indices[0]):
        if idx == -1:
            continue
        results.append(
            {
                "index": int(idx),
                "score": float(score),
                "text": store["texts"][idx],
            }
        )
    return results

print(f"✓ Model loaded: {MODEL_NAME}")
print(f"✓ Embedding model loaded: {EMBEDDING_MODEL_NAME}")
print(f"  GPU memory used: {torch.cuda.memory_allocated() / 1024**3:.1f} GB")

## Step 1 — Helpers and artifact directory

We start with simple notebook-native utilities. The benchmark stays intentionally small and explicit so every scoring rule is inspectable.

In [ ]:
from collections import defaultdict, Counter
import json
import math
import re
import statistics

random.seed(22)

ARTIFACT_DIR = Path("artifacts") / "notebook_22_castalia_bench"
ARTIFACT_DIR.mkdir(parents=True, exist_ok=True)


def normalize(text):
    text = str(text).lower()
    text = re.sub(r"[^a-z0-9\s]", " ", text)
    return re.sub(r"\s+", " ", text).strip()


def preview(text, width=88):
    text = str(text).replace("\n", " ").strip()
    return text if len(text) <= width else text[: width - 3] + "..."


def approx_tokens(text):
    return max(1, math.ceil(len(str(text)) / 4))


def to_markdown_table(rows, columns):
    header = "| " + " | ".join(columns) + " |"
    divider = "| " + " | ".join(["---"] * len(columns)) + " |"
    body = [
        "| " + " | ".join(str(row.get(column, "")) for column in columns) + " |"
        for row in rows
    ]
    return "\n".join([header, divider, *body])


print("Artifacts:", ARTIFACT_DIR.resolve())

## Step 2 — Build a compact local corpus

The corpus stays fully local and reproducible. Each document is short, factual, and easy to inspect, which makes later debugging much easier.

In [ ]:
CORPUS = [
    {
        "doc_id": "D1",
        "title": "Harbor District Microgrid Pilot",
        "text": "The Harbor district launched a resilience pilot in 2024. It installed 6 megawatts of rooftop solar and a 20 megawatt-hour battery. During two grid outages, the microgrid kept the health clinic and water pumps online for 11 hours. Operators reported a 34 percent reduction in diesel generator use compared with the previous summer.",
    },
    {
        "doc_id": "D2",
        "title": "Cedar Wastewater Heat Recovery Memo",
        "text": "The Cedar treatment plant now captures heat from outgoing effluent. The recovered heat keeps digesters warm and supplies a nearby greenhouse loop. Plant managers reported an 18 percent reduction in natural gas consumption. The memo estimates a payback period of about 5.5 years.",
    },
    {
        "doc_id": "D3",
        "title": "Northwind Offshore Wind Siting Brief",
        "text": "The Northwind team shifted turbine placement 9 kilometers east to avoid a major bird migration corridor. The export cable shares an existing shipping channel for the first 14 kilometers. Environmental review says construction noise is the largest short-term risk. The project brief estimates annual output could power 210000 homes.",
    },
    {
        "doc_id": "D4",
        "title": "Library Retrofit Case Study",
        "text": "A central library replaced chillers with ground-source heat pumps and smart ventilation controls. Energy use intensity fell from 212 to 148 kilowatt-hours per square meter. Summer comfort complaints dropped by 40 percent. Total project cost was 4.2 million dollars with an 11-year payback.",
    },
    {
        "doc_id": "D5",
        "title": "Drought Response Handbook",
        "text": "Stage two restrictions begin when reservoir storage stays below 62 percent for 21 consecutive days. The handbook says leak repairs can save up to 12 percent of summer demand. Recharge basins should rest every seventh day to prevent compaction. Public messaging focuses on irrigation schedules and the repair hotline.",
    },
    {
        "doc_id": "D6",
        "title": "Transit Electrification Plan",
        "text": "Phase one purchases 18 battery buses by 2026. The full program targets 42 electric buses by 2028. Overnight depot charging serves most vehicles, while an on-route pantograph at Central Station handles the peak corridor. The plan forecasts 22 percent lower maintenance cost than diesel operations.",
    },
    {
        "doc_id": "D7",
        "title": "Data Center Water Reuse Project",
        "text": "A reclaim system now reuses 68 percent of cooling water at the municipal data center. The facility pairs with treated greywater from the wastewater plant. Summer potable water demand fell by 41 percent after deployment. Filtration membranes are replaced every 18 months.",
    },
    {
        "doc_id": "D8",
        "title": "Estuary Flood Barrier Memo",
        "text": "The estuary barrier closes only during storm-surge warnings above 1.7 meters. Wetlands restoration is expected to delay peak flooding by 35 minutes. The annual maintenance budget is 8.4 million dollars. Fishing access is preserved through a side channel.",
    },
]

DOC_LOOKUP = {doc["doc_id"]: doc for doc in CORPUS}

corpus_rows = [
    {
        "doc_id": doc["doc_id"],
        "title": doc["title"],
        "words": len(doc["text"].split()),
    }
    for doc in CORPUS
]

print(to_markdown_table(corpus_rows, ["doc_id", "title", "words"]))

## Step 3 — Define the benchmark tasks

We use multiple task types so the benchmark is not biased toward a single system pattern. Some tasks are simple fact retrieval, some require synthesis, some require arithmetic, and one asks for a grounded recommendation.

In [ ]:
BENCHMARK = [
    {
        "task_id": "CB1",
        "task_type": "factoid",
        "question": "What storage capacity did the Harbor microgrid include, and how long did it keep the clinic and water pumps online during outages?",
        "required_docs": ["D1"],
        "checks": [
            ["20 megawatt hour", "20 mwh", "20 megawatt hour battery"],
            ["11 hours", "11 hour"],
        ],
    },
    {
        "task_id": "CB2",
        "task_type": "synthesis",
        "question": "Which two projects reduced natural gas or potable water demand through reuse or recovery, and by what percentages?",
        "required_docs": ["D2", "D7"],
        "checks": [
            ["cedar treatment plant", "cedar heat recovery", "cedar wastewater"],
            ["18 percent", "18%"],
            ["data center water reuse", "municipal data center", "water reuse project"],
            ["41 percent", "41%"],
        ],
    },
    {
        "task_id": "CB3",
        "task_type": "calculation",
        "question": "The library retrofit lowered energy use intensity from 212 to 148 kilowatt-hours per square meter. What was the absolute drop, and what approximate percent reduction is that?",
        "required_docs": ["D4"],
        "checks": [
            ["64"],
            ["30.2 percent", "30 percent", "30.19 percent"],
        ],
        "calculator_expressions": [
            {"label": "absolute_drop", "expression": "212 - 148"},
            {"label": "percent_reduction", "expression": "((212 - 148) / 212) * 100"},
        ],
    },
    {
        "task_id": "CB4",
        "task_type": "policy",
        "question": "According to the drought handbook, when do stage two restrictions begin, and how often should recharge basins rest?",
        "required_docs": ["D5"],
        "checks": [
            ["62 percent"],
            ["21 consecutive days", "21 days"],
            ["every seventh day", "seventh day", "every 7th day"],
        ],
    },
    {
        "task_id": "CB5",
        "task_type": "calculation",
        "question": "Transit plans buy 18 battery buses by 2026 and target 42 by 2028. How many additional buses are added after phase one, and what charging method handles the peak corridor?",
        "required_docs": ["D6"],
        "checks": [
            ["24"],
            ["pantograph", "central station"],
        ],
        "calculator_expressions": [
            {"label": "additional_buses", "expression": "42 - 18"},
        ],
    },
    {
        "task_id": "CB6",
        "task_type": "recommendation",
        "question": "A city wants resilience during outages and lower generator use. Based on the corpus, which project is the best fit and why?",
        "required_docs": ["D1"],
        "checks": [
            ["harbor district microgrid", "harbor microgrid", "microgrid pilot"],
            ["11 hours", "clinic and water pumps", "kept critical services online"],
            ["34 percent reduction in diesel", "34 percent diesel reduction", "34 percent lower diesel"],
        ],
    },
]

TASK_LOOKUP = {task["task_id"]: task for task in BENCHMARK}

task_rows = [
    {
        "task_id": task["task_id"],
        "type": task["task_type"],
        "required_docs": ", ".join(task["required_docs"]),
        "question": preview(task["question"], 92),
    }
    for task in BENCHMARK
]

print(to_markdown_table(task_rows, ["task_id", "type", "required_docs", "question"]))

## Step 4 — Chunk the corpus and build a tiny FAISS store

This is the same first-principles RAG pattern from earlier notebooks: chunk local text, embed it, index it, and keep the retrieval objects transparent.

In [ ]:
def split_sentences(text):
    cleaned = str(text).replace("\n", " ").strip()
    pieces = re.split(r"(?<=[.!?])\s+", cleaned)
    return [piece.strip() for piece in pieces if piece.strip()]


def make_chunks(corpus, sentences_per_chunk=2, overlap=1):
    chunks = []
    for doc in corpus:
        sentences = split_sentences(doc["text"])
        step = max(1, sentences_per_chunk - overlap)
        for start in range(0, len(sentences), step):
            window = sentences[start : start + sentences_per_chunk]
            if not window:
                continue
            chunks.append(
                {
                    "chunk_id": f'{doc["doc_id"]}_c{len(chunks):02d}',
                    "doc_id": doc["doc_id"],
                    "title": doc["title"],
                    "text": " ".join(window),
                }
            )
            if start + sentences_per_chunk >= len(sentences):
                break
    return chunks


CHUNKS = make_chunks(CORPUS, sentences_per_chunk=2, overlap=1)
CHUNK_TEXTS = [
    f'{chunk["doc_id"]} | {chunk["title"]} | {chunk["text"]}'
    for chunk in CHUNKS
]
VECTOR_STORE = build_faiss_index(CHUNK_TEXTS)


def search_corpus(query, k=4):
    matches = search_index(query, VECTOR_STORE, k=k)
    rows = []
    for match in matches:
        chunk = CHUNKS[match["index"]]
        rows.append(
            {
                "chunk_id": chunk["chunk_id"],
                "doc_id": chunk["doc_id"],
                "title": chunk["title"],
                "score": round(match["score"], 3),
                "text": chunk["text"],
            }
        )
    return rows


sample_hits = search_corpus("generator reduction during outages", k=3)
print("Chunks:", len(CHUNKS))
print(to_markdown_table(sample_hits, ["chunk_id", "doc_id", "score", "title"]))

## Step 5 — Shared prompting and parsing helpers

All systems will answer in the same simple format so the harness can score them with consistent parsers.

In [ ]:
ANSWER_FORMAT_INSTRUCTIONS = """You are participating in Castalia Bench.
Answer the question briefly and clearly.
Use this exact format:
Answer: <your answer>
Evidence: <comma-separated document ids like D1, D2, or 'none'>

Only cite document ids that genuinely support the answer.
If you lack supporting evidence, write: Evidence: none
"""


def format_context(results, max_chars=1500):
    parts = []
    used = 0
    for result in results:
        block = f'[{result["doc_id"]}] {result["title"]}: {result["text"]}'
        if used + len(block) > max_chars and parts:
            break
        parts.append(block)
        used += len(block)
    return "\n".join(parts)


def parse_response(text):
    text = str(text).strip()
    answer_match = re.search(r"answer:\s*(.*?)(?:\n\s*evidence:|\Z)", text, flags=re.IGNORECASE | re.DOTALL)
    evidence_match = re.search(r"evidence:\s*(.*)", text, flags=re.IGNORECASE)
    answer = answer_match.group(1).strip() if answer_match else text
    evidence_text = evidence_match.group(1).strip() if evidence_match else "none"
    cited_docs = sorted(set(re.findall(r"D\d+", evidence_text.upper())))
    return {
        "answer": answer,
        "evidence_text": evidence_text,
        "cited_docs": cited_docs,
    }


def timed_generate(messages, max_new_tokens=220):
    start = time.perf_counter()
    raw_output = generate(messages, max_new_tokens=max_new_tokens, temperature=0.0, do_sample=False)
    latency_s = round(time.perf_counter() - start, 3)
    prompt_text = json.dumps(messages, ensure_ascii=False)
    return {
        "raw_output": raw_output,
        "latency_s": latency_s,
        "prompt_tokens": approx_tokens(prompt_text),
        "output_tokens": approx_tokens(raw_output),
    }

## Step 6 — System under test 1: prompt-only baseline

This baseline gets the benchmark question and formatting instructions, but no retrieved context and no tools.

In [ ]:
def run_prompt_baseline(task):
    messages = [
        {"role": "system", "content": ANSWER_FORMAT_INSTRUCTIONS},
        {
            "role": "user",
            "content": (
                "System type: prompt-only baseline.\n"
                "You do not have access to external documents or tools.\n"
                f'Task type: {task["task_type"]}\n'
                f'Question: {task["question"]}'
            ),
        },
    ]
    generation = timed_generate(messages)
    parsed = parse_response(generation["raw_output"])
    return {
        "task_id": task["task_id"],
        "task_type": task["task_type"],
        "system_name": "prompt_only",
        "question": task["question"],
        "answer": parsed["answer"],
        "raw_output": generation["raw_output"],
        "cited_docs": parsed["cited_docs"],
        "evidence_text": parsed["evidence_text"],
        "retrieved_docs": [],
        "retrieved_chunks": 0,
        "retrieval_calls": 0,
        "tool_calls": 0,
        "tool_trace": [],
        "prompt_tokens": generation["prompt_tokens"],
        "output_tokens": generation["output_tokens"],
        "latency_s": generation["latency_s"],
    }

## Step 7 — System under test 2: tiny RAG pipeline

The RAG system retrieves local chunks with embeddings + FAISS, passes those chunks to the model, and asks for a grounded answer with document citations.

In [ ]:
def run_rag_pipeline(task, k=4):
    retrieved = search_corpus(task["question"], k=k)
    context = format_context(retrieved)
    messages = [
        {"role": "system", "content": ANSWER_FORMAT_INSTRUCTIONS},
        {
            "role": "user",
            "content": (
                "System type: local RAG pipeline.\n"
                "Answer only from the provided context when possible.\n"
                f'Task type: {task["task_type"]}\n'
                f'Question: {task["question"]}\n\n'
                "Context:\n"
                f"{context}"
            ),
        },
    ]
    generation = timed_generate(messages)
    parsed = parse_response(generation["raw_output"])
    return {
        "task_id": task["task_id"],
        "task_type": task["task_type"],
        "system_name": "rag_pipeline",
        "question": task["question"],
        "answer": parsed["answer"],
        "raw_output": generation["raw_output"],
        "cited_docs": parsed["cited_docs"],
        "evidence_text": parsed["evidence_text"],
        "retrieved_docs": sorted({row["doc_id"] for row in retrieved}),
        "retrieved_chunks": len(retrieved),
        "retrieval_calls": 1,
        "tool_calls": 0,
        "tool_trace": [
            {
                "tool": "search_corpus",
                "query": task["question"],
                "top_doc_ids": [row["doc_id"] for row in retrieved],
            }
        ],
        "prompt_tokens": generation["prompt_tokens"],
        "output_tokens": generation["output_tokens"],
        "latency_s": generation["latency_s"],
    }

## Step 8 — System under test 3: simple tool-using agent

The agent stays intentionally small:

1. search the local corpus
2. optionally use a calculator for arithmetic tasks
3. hand the gathered observations to the model for the final answer

This is enough to show what a tool-using workflow buys over plain prompting and plain retrieval.

In [ ]:
SAFE_EXPR = re.compile(r"^[0-9\.\+\-\*\/\(\) ]+$")


def calculator_tool(expression):
    expression = str(expression).strip()
    if not SAFE_EXPR.fullmatch(expression):
        raise ValueError(f"Unsafe expression: {expression}")
    return eval(expression, {"__builtins__": {}}, {})


def run_agent_workflow(task, k=5):
    trace = []

    retrieved = search_corpus(task["question"], k=k)
    trace.append(
        {
            "tool": "search_corpus",
            "query": task["question"],
            "top_doc_ids": [row["doc_id"] for row in retrieved],
        }
    )

    calculator_results = []
    for item in task.get("calculator_expressions", []):
        value = calculator_tool(item["expression"])
        if isinstance(value, float):
            value = round(value, 2)
        calculator_results.append({"label": item["label"], "value": value})
        trace.append(
            {
                "tool": "calculator",
                "expression": item["expression"],
                "value": value,
            }
        )

    context = format_context(retrieved)
    tool_notes = []
    if calculator_results:
        tool_notes.append("Calculator results:")
        for item in calculator_results:
            tool_notes.append(f'- {item["label"]}: {item["value"]}')
    else:
        tool_notes.append("Calculator results: none")

    messages = [
        {"role": "system", "content": ANSWER_FORMAT_INSTRUCTIONS},
        {
            "role": "user",
            "content": (
                "System type: simple tool-using agent.\n"
                "Use the search results and calculator observations below.\n"
                "Prefer those observations over guessing.\n"
                f'Task type: {task["task_type"]}\n'
                f'Question: {task["question"]}\n\n'
                "Retrieved context:\n"
                f"{context}\n\n"
                + "\n".join(tool_notes)
            ),
        },
    ]

    generation = timed_generate(messages)
    parsed = parse_response(generation["raw_output"])
    return {
        "task_id": task["task_id"],
        "task_type": task["task_type"],
        "system_name": "agent_workflow",
        "question": task["question"],
        "answer": parsed["answer"],
        "raw_output": generation["raw_output"],
        "cited_docs": parsed["cited_docs"],
        "evidence_text": parsed["evidence_text"],
        "retrieved_docs": sorted({row["doc_id"] for row in retrieved}),
        "retrieved_chunks": len(retrieved),
        "retrieval_calls": 1,
        "tool_calls": len(trace),
        "tool_trace": trace,
        "calculator_results": calculator_results,
        "prompt_tokens": generation["prompt_tokens"],
        "output_tokens": generation["output_tokens"],
        "latency_s": generation["latency_s"],
    }

## Step 9 — Run all three systems on the same benchmark

This is the core capstone move: one dataset, one harness, three system designs.

In [ ]:
SYSTEMS = {
    "prompt_only": run_prompt_baseline,
    "rag_pipeline": run_rag_pipeline,
    "agent_workflow": run_agent_workflow,
}

raw_runs = []
for task in BENCHMARK:
    for system_name, runner in SYSTEMS.items():
        print(f'Running {system_name} on {task["task_id"]}...')
        raw_runs.append(runner(task))

preview_rows = [
    {
        "task_id": row["task_id"],
        "system": row["system_name"],
        "cited_docs": ", ".join(row["cited_docs"]) or "none",
        "latency_s": row["latency_s"],
        "answer": preview(row["answer"], 74),
    }
    for row in raw_runs
]

print("Total runs:", len(raw_runs))
print(to_markdown_table(preview_rows, ["task_id", "system", "cited_docs", "latency_s", "answer"]))

## Step 10 — Build scorers for quality, groundedness, cost proxy, and success

The benchmark uses deliberately plain scoring rules:

- **quality**: point-based fact or rubric coverage
- **groundedness**: required evidence recall plus citation precision
- **latency**: measured wall-clock generation time
- **cost proxy**: approximate token + retrieval + tool burden
- **success**: quality and groundedness both clear a minimum bar

In [ ]:
def matches_alternative(answer, alternative):
    answer_tokens = set(normalize(answer).split())
    alt_tokens = normalize(alternative).split()
    return bool(alt_tokens) and all(token in answer_tokens for token in alt_tokens)


def score_quality(task, answer):
    hits = 0
    for alternatives in task["checks"]:
        if any(matches_alternative(answer, alternative) for alternative in alternatives):
            hits += 1
    total = len(task["checks"])
    return round(hits / total, 3), hits, total


def score_groundedness(task, cited_docs):
    required = set(task["required_docs"])
    cited = set(cited_docs)
    if not cited:
        return 0.0, 0.0, 0.0
    overlap = required & cited
    citation_recall = len(overlap) / len(required) if required else 1.0
    citation_precision = len(overlap) / len(cited) if cited else 0.0
    groundedness = round(0.7 * citation_recall + 0.3 * citation_precision, 3)
    return round(citation_recall, 3), round(citation_precision, 3), groundedness


def score_retrieval_hit(task, retrieved_docs):
    required = set(task["required_docs"])
    if not required:
        return 1.0
    return round(len(required & set(retrieved_docs)) / len(required), 3)


def cost_proxy(run):
    token_cost = (run["prompt_tokens"] + run["output_tokens"]) / 1000
    retrieval_cost = 0.05 * run["retrieval_calls"] + 0.01 * run["retrieved_chunks"]
    tool_cost = 0.08 * run["tool_calls"]
    return round(token_cost + retrieval_cost + tool_cost, 3)


def success_flag(row):
    return row["quality_score"] >= 0.67 and row["groundedness_score"] >= 0.5

## Step 11 — Score every run

We now attach benchmark metrics to each response so later tables and diagnostics stay consistent.

In [ ]:
scored_runs = []
for run in raw_runs:
    task = TASK_LOOKUP[run["task_id"]]
    quality_score, quality_hits, quality_total = score_quality(task, run["answer"])
    citation_recall, citation_precision, groundedness_score = score_groundedness(task, run["cited_docs"])
    retrieval_hit = score_retrieval_hit(task, run["retrieved_docs"])

    row = {
        **run,
        "required_docs": task["required_docs"],
        "quality_score": quality_score,
        "quality_hits": quality_hits,
        "quality_total": quality_total,
        "citation_recall": citation_recall,
        "citation_precision": citation_precision,
        "groundedness_score": groundedness_score,
        "retrieval_hit": retrieval_hit,
        "cost_proxy": cost_proxy(run),
    }
    row["success"] = success_flag(row)
    scored_runs.append(row)

score_rows = [
    {
        "task_id": row["task_id"],
        "system": row["system_name"],
        "quality": row["quality_score"],
        "grounded": row["groundedness_score"],
        "success": row["success"],
        "latency_s": row["latency_s"],
        "cost_proxy": row["cost_proxy"],
    }
    for row in scored_runs
]

print(to_markdown_table(score_rows, ["task_id", "system", "quality", "grounded", "success", "latency_s", "cost_proxy"]))

## Step 12 — Side-by-side system comparison

This is the first summary view you would want in a real benchmark report.

In [ ]:
grouped_runs = defaultdict(list)
for row in scored_runs:
    grouped_runs[row["system_name"]].append(row)

summary_rows = []
for system_name, rows in grouped_runs.items():
    summary_rows.append(
        {
            "system": system_name,
            "avg_quality": round(statistics.mean(r["quality_score"] for r in rows), 3),
            "avg_groundedness": round(statistics.mean(r["groundedness_score"] for r in rows), 3),
            "success_rate": round(statistics.mean(1 if r["success"] else 0 for r in rows), 3),
            "avg_latency_s": round(statistics.mean(r["latency_s"] for r in rows), 3),
            "avg_cost_proxy": round(statistics.mean(r["cost_proxy"] for r in rows), 3),
            "avg_retrieval_hit": round(statistics.mean(r["retrieval_hit"] for r in rows), 3),
            "avg_tool_calls": round(statistics.mean(r["tool_calls"] for r in rows), 2),
        }
    )

summary_rows = sorted(summary_rows, key=lambda row: (row["success_rate"], row["avg_quality"]), reverse=True)
print(to_markdown_table(summary_rows, [
    "system",
    "avg_quality",
    "avg_groundedness",
    "success_rate",
    "avg_latency_s",
    "avg_cost_proxy",
    "avg_retrieval_hit",
    "avg_tool_calls",
]))

## Step 13 — Compare systems by task

A benchmark should reveal not only the global winner, but also where each design wins.

In [ ]:
task_comparison_rows = []
for task in BENCHMARK:
    rows = [row for row in scored_runs if row["task_id"] == task["task_id"]]
    best = max(rows, key=lambda row: (row["success"], row["quality_score"], row["groundedness_score"]))
    lookup = {row["system_name"]: row for row in rows}
    task_comparison_rows.append(
        {
            "task_id": task["task_id"],
            "type": task["task_type"],
            "prompt_quality": lookup["prompt_only"]["quality_score"],
            "rag_quality": lookup["rag_pipeline"]["quality_score"],
            "agent_quality": lookup["agent_workflow"]["quality_score"],
            "prompt_grounded": lookup["prompt_only"]["groundedness_score"],
            "rag_grounded": lookup["rag_pipeline"]["groundedness_score"],
            "agent_grounded": lookup["agent_workflow"]["groundedness_score"],
            "winner": best["system_name"],
        }
    )

print(to_markdown_table(task_comparison_rows, [
    "task_id",
    "type",
    "prompt_quality",
    "rag_quality",
    "agent_quality",
    "prompt_grounded",
    "rag_grounded",
    "agent_grounded",
    "winner",
]))

## Step 14 — Add failure buckets

Failure buckets turn raw scores into engineering guidance. We want to know *how* systems fail, not just that they fail.

In [ ]:
def assign_failure_buckets(row):
    task = TASK_LOOKUP[row["task_id"]]
    buckets = []

    if row["quality_score"] < 0.67:
        if task["task_type"] == "calculation":
            buckets.append("calculation_miss")
        elif task["task_type"] == "recommendation":
            buckets.append("weak_recommendation")
        elif task["task_type"] == "synthesis":
            buckets.append("incomplete_synthesis")
        else:
            buckets.append("fact_coverage_miss")

    if row["groundedness_score"] == 0:
        buckets.append("missing_evidence")
    elif row["citation_precision"] < 1.0:
        buckets.append("wrong_or_extra_citation")

    if row["retrieval_calls"] > 0 and row["retrieval_hit"] < 1.0:
        buckets.append("retrieval_miss")

    if row["tool_calls"] > 0 and task["task_type"] == "calculation" and row["quality_score"] < 1.0:
        buckets.append("tooling_did_not_rescue_math")

    return buckets or ["passed"]


for row in scored_runs:
    row["failure_buckets"] = assign_failure_buckets(row)

bucket_counter = Counter()
for row in scored_runs:
    for bucket in row["failure_buckets"]:
        if bucket != "passed":
            bucket_counter[(row["system_name"], bucket)] += 1

bucket_rows = [
    {"system": system, "bucket": bucket, "count": count}
    for (system, bucket), count in sorted(bucket_counter.items())
]

failed_examples = [
    {
        "task_id": row["task_id"],
        "system": row["system_name"],
        "buckets": ", ".join(row["failure_buckets"]),
        "answer": preview(row["answer"], 88),
    }
    for row in scored_runs
    if "passed" not in row["failure_buckets"]
]

print(to_markdown_table(bucket_rows, ["system", "bucket", "count"]))
print()
print(to_markdown_table(failed_examples, ["task_id", "system", "buckets", "answer"]))

## Step 15 — Evidence and tool diagnostics

These extra slices help explain *why* the summary table looks the way it does.

In [ ]:
diagnostic_rows = []
for system_name, rows in grouped_runs.items():
    diagnostic_rows.append(
        {
            "system": system_name,
            "citation_recall": round(statistics.mean(r["citation_recall"] for r in rows), 3),
            "citation_precision": round(statistics.mean(r["citation_precision"] for r in rows), 3),
            "retrieval_hit": round(statistics.mean(r["retrieval_hit"] for r in rows), 3),
            "avg_prompt_tokens": round(statistics.mean(r["prompt_tokens"] for r in rows), 1),
            "avg_output_tokens": round(statistics.mean(r["output_tokens"] for r in rows), 1),
        }
    )

agent_math_trace = [
    row for row in scored_runs
    if row["system_name"] == "agent_workflow" and row["task_type"] == "calculation"
][0]

print(to_markdown_table(diagnostic_rows, [
    "system",
    "citation_recall",
    "citation_precision",
    "retrieval_hit",
    "avg_prompt_tokens",
    "avg_output_tokens",
]))

print()
print("Sample agent trace for", agent_math_trace["task_id"])
print(json.dumps(agent_math_trace["tool_trace"], indent=2))

## Step 16 — Ask the local model to write the benchmark report

The report is generated *after* the metrics exist. This keeps the narrative grounded in measured results instead of vibes.

In [ ]:
report_prompt = f"""
You are writing a concise benchmark report for the Castalia Bench capstone.

System summary:
{json.dumps(summary_rows, indent=2)}

Task comparison:
{json.dumps(task_comparison_rows, indent=2)}

Failure buckets:
{json.dumps(bucket_rows, indent=2)}

Write a short report with:
1. the overall winner
2. what prompting buys
3. what RAG buys
4. what the agent buys
5. one next-step recommendation
Keep it concise and concrete.
"""

benchmark_report = generate(report_prompt, max_new_tokens=260, temperature=0.0, do_sample=False)
print(benchmark_report)

## Step 17 — Save benchmark artifacts

The benchmark is more useful when its dataset, run logs, summaries, and report survive beyond one notebook session.

In [ ]:
dataset_artifact = {
    "corpus": CORPUS,
    "benchmark": BENCHMARK,
}

summary_artifact = {
    "summary_rows": summary_rows,
    "task_comparison_rows": task_comparison_rows,
    "bucket_rows": bucket_rows,
    "diagnostic_rows": diagnostic_rows,
}

(ARTIFACT_DIR / "dataset.json").write_text(json.dumps(dataset_artifact, indent=2), encoding="utf-8")
(ARTIFACT_DIR / "raw_runs.json").write_text(json.dumps(raw_runs, indent=2), encoding="utf-8")
(ARTIFACT_DIR / "scored_runs.json").write_text(json.dumps(scored_runs, indent=2), encoding="utf-8")
(ARTIFACT_DIR / "summary.json").write_text(json.dumps(summary_artifact, indent=2), encoding="utf-8")
(ARTIFACT_DIR / "benchmark_report.txt").write_text(benchmark_report.strip() + "\n", encoding="utf-8")

leaderboard_md = to_markdown_table(summary_rows, [
    "system",
    "avg_quality",
    "avg_groundedness",
    "success_rate",
    "avg_latency_s",
    "avg_cost_proxy",
    "avg_retrieval_hit",
    "avg_tool_calls",
])
(ARTIFACT_DIR / "leaderboard.md").write_text(leaderboard_md, encoding="utf-8")

print("Saved files:")
for path in sorted(ARTIFACT_DIR.iterdir()):
    print("-", path.name)

## Conclusion

Castalia Bench shows the course thesis in one place:

- **Prompting** is the cheapest and simplest layer, but it struggles when precise local evidence matters.
- **RAG** adds grounded recall and usually becomes the best default for factual local tasks.
- **Agents** add extra control when tasks need tools, intermediate steps, or deterministic operations like calculation.
- **Evals** sits above all three because it is the layer that tells you when the extra complexity is actually worth it.

That is the point of this course: not merely building prompt systems, RAG systems, and agents, but measuring them with enough discipline to improve them on purpose.

## 🏋️ Exercises

Put your understanding to the test:

**Exercise 1 — Explore:** design an eval dataset for a new use case. Document what changes and why.

**Exercise 2 — Build:** implement a custom metric and compare it to the one in this notebook. Compare your implementation to the one in this notebook.

**Exercise 3 — Extend:** run the evaluation on a different model and analyze the differences.

## 📚 References & Further Reading

- [Hugging Face Documentation](https://huggingface.co/docs)
- [LLM Course Resources](https://github.com/cyruslayo/castalia)
- Explore related notebooks in the evals/ directory

---

## 🧭 Navigation

⬅️ **Previous:** [21 Experiment Tracking And Reporting](https://colab.research.google.com/github/cyruslayo/castalia/blob/main/evals/21_experiment_tracking_and_reporting.ipynb)